# ASTERIX notebook: loading blocks, splitting messages, and detecting Data Items from FSPEC

This notebook is a teaching scaffold, not a full production decoder.

It is organized in three stages:

1. **Load the raw binary file**
2. **Split the file into ASTERIX messages using `CAT + LEN`**
3. **Read the FSPEC of each message and map detected FRNs to Data Items using the CSV tables**

Important note:

- This notebook assumes **one record per ASTERIX message block** for the FSPEC detection stage.
- That is enough to learn the message structure and see which Data Items are announced by the FSPEC.
- A full multi-record decoder would also need to decode the length of every item inside each record, especially the variable-length and compound items.


## Block 1 — Load the binary file

This block opens the `.ast` binary file and prints a quick preview.

What this block should show:

- total number of bytes loaded
- first bytes in hexadecimal
- a reminder that byte 1 is `CAT` and bytes 2-3 are `LEN`

By default, the notebook points to a small demo file created for testing.  
Replace the file path with your real `.ast` file when you are ready.


In [ ]:
from pathlib import Path

# Path to the ASTERIX binary file.
# Replace this with your own file when you want to test real data.
BINARY_PATH = Path("raw_data/datos_asterix_adsb.ast")

# Read the full file as raw bytes.
raw_data = BINARY_PATH.read_bytes()

# Print a small preview to confirm the file was loaded correctly.
print(f"Loaded file: {BINARY_PATH}")
print(f"Total bytes loaded: {len(raw_data)}")
print("First 32 bytes (hex):", raw_data[:32].hex(" ").upper())

# Show the first message header if enough bytes are available.
if len(raw_data) >= 3:
    first_cat = raw_data[0]
    first_len = int.from_bytes(raw_data[1:3], byteorder="big")
    print(f"First CAT value: {first_cat} (0x{first_cat:02X})")
    print(f"First LEN value: {first_len} bytes")
else:
    print("The file is too short to contain a valid ASTERIX header.")


## Block 2 — Split the binary into ASTERIX messages

Every ASTERIX message block starts with:

- **Octet 1**: `CAT`
- **Octets 2-3**: `LEN` (big-endian)

This block walks through the raw binary buffer and slices it into complete messages.
Each message is stored in a list as a dictionary with:

- `index`
- `offset`
- `cat`
- `length`
- `raw_records`


In [ ]:
def split_asterix_messages(raw_bytes: bytes):
    """Split a binary ASTERIX stream into individual message blocks."""
    messages = []
    offset = 0
    message_index = 0

    while offset < len(raw_bytes):
        # At least 3 bytes are needed for CAT + LEN.
        if offset + 3 > len(raw_bytes):
            print(f"Stopped at offset {offset}: not enough bytes for CAT + LEN.")
            break

        cat = raw_bytes[offset]
        length = int.from_bytes(raw_bytes[offset + 1: offset + 3], byteorder="big")

        # Basic validation of the declared length.
        if length < 3:
            print(f"Stopped at offset {offset}: invalid LEN={length}.")
            break

        if offset + length > len(raw_bytes):
            print(
                f"Stopped at offset {offset}: LEN={length} goes beyond file size."
            )
            break

        message_bytes = raw_bytes[offset+3: offset + length]

        messages.append(
            {
                "index": message_index,
                "offset": offset,
                "cat": cat,
                "length": length,
                "raw_records": message_bytes,
            }
        )

        offset += length
        message_index += 1

    return messages


messages = split_asterix_messages(raw_data)

print(f"Total ASTERIX messages found: {len(messages)}")
for msg in messages:
    print(
        f"#{msg['index']} ({msg['offset']}) "
        f"CAT={msg['cat']} (0x{msg['cat']:02X}), LEN={msg['length']}, "
        f"raw_records={msg['raw_records'].hex(' ').upper()}"
    )


## Block 3 — Extract the first record FSPEC and print the FRNs

This block parses the **FSPEC of the first record only** inside each message.

Why only the first record?

Because after the FSPEC, the message contains Data Fields with mixed lengths (fixed, variable, repetitive, compound, etc.). Without fully decoding those fields, we cannot know where the next record starts. So this block is intentionally limited to the first record of each message.

In [ ]:
def parse_fspec_from_record_start(message_bytes: bytes):
    """Parse FSPEC starting immediately after CAT+LEN.
    
    Returns:
        fspec_octets: list of integers
        frns: list of FRN numbers present in the first record
    """
    fspec_octets = []
    frns = []
    frn_counter = 1
    cursor = 0

    while cursor < len(message_bytes):
        octet = message_bytes[cursor]
        fspec_octets.append(octet)

        # In FSPEC, bits 8..2 indicate seven FRNs, and bit 1 is FX.
        # We inspect the bits from MSB to bit-2 to preserve FRN order.
        for bit_index in range(7, 0, -1):
            bit_value = (octet >> bit_index) & 0b1
            if bit_value == 1:
                frns.append(frn_counter)
            frn_counter += 1

        fx = octet & 0b1
        cursor += 1
        if fx == 0:
            break

    return fspec_octets, frns


message_fspec_info = []

for msg in messages:
    fspec_octets, frns = parse_fspec_from_record_start(msg['raw_records'])
    fspec_hex = ' '.join(f'{b:02X}' for b in fspec_octets)
    frn_labels = ', '.join(f'FRN{n}' for n in frns) if frns else '(no FRNs detected)'

    message_fspec_info.append({
        'index': msg['index'],
        'cat': msg['cat'],
        'fspec_octets': fspec_octets,
        'frns': frns,
    })

    print(f"#{msg['index']} FSPEC={frn_labels} ({fspec_hex})")


## Block 4 — Build an FRN presence matrix and map to Data Items via UAP tables

Instead of looping message by message, this block:

1. **Builds a binary matrix** (`DataFrame`) where each row is a message and each column is an FRN. A cell contains `1` if that FRN is present in the first-record FSPEC, `0` otherwise.
2. **Loads the UAP CSV tables once** and creates a fast `{FRN → Data Item, Description}` lookup per CAT.
3. **Renames the columns** from plain FRN numbers to `"FRN{n}: {Data Item} — {Description}"` so the table is self-documenting.

The result is a single DataFrame per CAT that you can filter, export, or visualize instantly.


In [ ]:
import pandas as pd

# ── 1. Load UAP tables and build fast FRN → label lookups ──────────────────

cat021_uap = pd.read_csv('asterix_decoder/uap_tables/CAT021.csv')
cat048_uap = pd.read_csv('asterix_decoder/uap_tables/CAT048.csv')

def _build_frn_lookup(uap_df: pd.DataFrame) -> dict:
    """Return {frn_int: 'Data Item — Description'} skipping FX rows."""
    lookup = {}
    for _, row in uap_df.iterrows():
        raw = row.get('FRN')
        if pd.isna(raw):
            continue
        text = str(raw).strip().upper()
        if text in ('FX', '-'):
            continue
        try:
            frn = int(float(text))
        except ValueError:
            continue
        data_item = str(row.get('Data Item', '')).strip()
        desc = str(row.get('Description', row.get('Data Item Description', ''))).strip()
        lookup[frn] = f"{data_item} — {desc}"
    return lookup

uap_lookups = {
    21: _build_frn_lookup(cat021_uap),
    48: _build_frn_lookup(cat048_uap),
}

# ── 2. Group messages by CAT ───────────────────────────────────────────────

from collections import defaultdict

cat_groups = defaultdict(list)          # {cat: [(msg_index, frns), ...]}
for info in message_fspec_info:
    cat_groups[info['cat']].append((info['index'], info['frns']))

# ── 3. Build one FRN presence matrix per CAT and rename columns ────────────

frn_matrices = {}                       # {cat: DataFrame}

for cat, entries in cat_groups.items():
    lookup = uap_lookups.get(cat, {})

    # Determine all FRNs that appear in any message of this CAT.
    all_frns = sorted({frn for _, frns in entries for frn in frns})

    # Build the binary matrix (one row per message, one col per FRN).
    rows = []
    msg_indices = []
    for msg_idx, frns in entries:
        frn_set = set(frns)
        rows.append({frn: int(frn in frn_set) for frn in all_frns})
        msg_indices.append(msg_idx)

    df = pd.DataFrame(rows, index=pd.Index(msg_indices, name='Message #'))

    # Rename integer FRN columns to descriptive labels.
    col_rename = {}
    for frn in all_frns:
        label = lookup.get(frn)
        if label:
            col_rename[frn] = f"FRN{frn}: {label}"
        else:
            col_rename[frn] = f"FRN{frn}: (unknown)"
    df.rename(columns=col_rename, inplace=True)

    frn_matrices[cat] = df

# ── 4. Display results per message ─────────────────────────────────────────

for cat, entries in cat_groups.items():
    lookup = uap_lookups.get(cat, {})
    
    for msg_idx, frns in entries:
        # Map each FRN to its Data Item code (e.g., "I021/010")
        data_items = []
        for frn in sorted(frns):
            label = lookup.get(frn)
            if label:
                # Extract just the Data Item code (first part before " — ")
                data_item_code = label.split(' — ')[0]
                data_items.append(data_item_code)
            else:
                data_items.append(f"I{cat:03d}/FRN{frn}")
        
        items_str = ' '.join(data_items)
        print(f"  #{msg_idx} DataItems({len(data_items)})= {items_str}")